# Lecture 4 Lab — Predictive Maintenance with Ensemble Methods
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours |
| **Tools** | Python, NumPy, pandas, matplotlib, scikit-learn, XGBoost |
| **Submit** | This completed notebook (see §10) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID


## 1. Learning Objectives

By the end of this lab, you should be able to:

- Train and compare bagging, random forest, gradient boosting, and XGBoost classifiers on the same task.
- Explain, using your own results, why boosting tends to outperform a single tree but needs more careful tuning.
- Extract and interpret feature importances from a tree-based ensemble for root-cause analysis.
- Choose an ensemble method for a predictive-maintenance task and justify the choice with accuracy and interpretability trade-offs.

## 2. Setup

This week adds one new package. If you haven't already:

```bash
pip install xgboost
```

(may result in error for 32-bit Python. Works fine with uv environment.)

In [ ]:
# pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay
import xgboost as xgb

print("Environment OK — xgboost", xgb.__version__)

**Note :** For all TODO's in this lab, you may specify random_state = 0 (or other value) to make sure your result is reproducible.

## 3. Part A — Predictive Maintenance Dataset

Three sensor-derived features (`vibration_rms`, `temperature`, `running_hours`) and a three-class target: `Low Risk`, `Medium Risk`, `High Risk`. Unlike Week 2, there's no missing data or categorical column to clean up this week — the focus is entirely on the models themselves.

In [ ]:
def generate_maintenance_dataset(n_per_class=200, seed=0):
    rng = np.random.default_rng(seed)
    class_names = ["Low Risk", "Medium Risk", "High Risk"]
    centers = {
        "Low Risk":    dict(vibration_rms=0.15, temperature=45, running_hours=800),
        "Medium Risk": dict(vibration_rms=0.35, temperature=60, running_hours=2500),
        "High Risk":   dict(vibration_rms=0.65, temperature=75, running_hours=4500),
    }
    rows = []
    for label_idx, cname in enumerate(class_names):
        c = centers[cname]
        for _ in range(n_per_class):
            rows.append({
                "vibration_rms": max(0.0, rng.normal(c["vibration_rms"], 0.09)),
                "temperature": rng.normal(c["temperature"], 6),
                "running_hours": max(0.0, rng.normal(c["running_hours"], 550)),
                "risk": label_idx,
                "risk_name": cname,
            })
    df = pd.DataFrame(rows)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


df = generate_maintenance_dataset()
df.head()

In [ ]:

X = df[["vibration_rms", "temperature", "running_hours"]]
y = df["risk"]
class_names = ["Low Risk", "Medium Risk", "High Risk"]
# TODO: create a 70% training and 30% testing samples from the dataset. Set random_state to 0, and make sure that each class 
# has equal number of samples
X_train, X_test, y_train, y_test = None
print(f"Train: {len(X_train)}   Test: {len(X_test)}")

## 4. Part B — Baseline: A Single Decision Tree

Your Week 2 baseline, for comparison against everything else this week.

In [ ]:
# TODO : create a decision tree classifier with maximum depth of 4, and fit on traning data.
tree = None
tree.fit(X_train, y_train)
tree_acc = accuracy_score(y_test, tree.predict(X_test))
print(f"Single Decision Tree   accuracy: {tree_acc:.3f}")

## 5. Part C — Bagging

Fifty trees, each on a bootstrap resample of the training data, combined by majority vote.

In [ ]:
# TODO : create a bagging classifier using 50 decision trees as estimators, and fit on traning data.
bagging = None
bagging.fit(X_train, y_train)
bagging_acc = accuracy_score(y_test, bagging.predict(X_test))
print(f"Bagging (50 trees)      accuracy: {bagging_acc:.3f}")

## 6. Part D — Random Forest + Feature Importance

Fit a random forest, then fill in the `TODO` to extract and plot its feature importances.

In [ ]:
 # TODO : create a random forest classifier with 200 estimators, maximum dept of 6, and fit on traning data.
rf = None
rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Random Forest            accuracy: {rf_acc:.3f}")

In [ ]:
# TODO: get the feature importances out of the fitted random forest.
# Hint: rf.feature_importances_ is an array in the same order as X.columns.

rf_importances = None


feat_names = list(X.columns)
plt.figure(figsize=(6, 3.5))
plt.barh(feat_names, rf_importances, color="#FF6A39")
plt.xlabel("Importance")
plt.title("Random Forest — feature importance")
plt.tight_layout()
plt.show()

## 7. Part E — Gradient Boosting & XGBoost

Fit scikit-learn's `GradientBoostingClassifier`, then fill in the `TODO` to build and fit an `xgb.XGBClassifier` with the same rough settings.

In [ ]:
 # TODO : create a gradient boosting classifier with 150 estimators, maximum dept of 6, learning rate = 0.1,
 # and fit on traning data.

gb = None
gb.fit(X_train, y_train)
gb_acc = accuracy_score(y_test, gb.predict(X_test))
print(f"Gradient Boosting        accuracy: {gb_acc:.3f}")

In [ ]:
# TODO: build an xgb.XGBClassifier with n_estimators=200, max_depth=4,
# learning_rate=0.1, and fit it on X_train, y_train.
xgb_model = None
xgb_model.fit(X_train, y_train)

xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test))
print(f"XGBoost                  accuracy: {xgb_acc:.3f}")

## 8. Part F — Final Comparison

Collect every model's test accuracy and macro-F1 (a fairer summary than accuracy alone when classes might be imbalanced), and compare them side by side.

In [ ]:
models = {
    "Decision Tree": tree,
    "Bagging": bagging,
    "Random Forest": rf,
    "Gradient Boosting": gb,
    "XGBoost": xgb_model,
}

results = {}
for name, model in models.items():
    preds = model.predict(X_test)
    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "f1_macro": f1_score(y_test, preds, average="macro"),
    }

results_df = pd.DataFrame(results).T
results_df

In [ ]:
results_df[["accuracy", "f1_macro"]].plot(kind="bar", figsize=(8, 4.5), color=["#3E5C76", "#FF6A39"])
plt.ylabel("Score")
plt.title("Ensemble comparison — predictive maintenance")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

best_name = results_df["f1_macro"].idxmax()
ConfusionMatrixDisplay.from_estimator(
    models[best_name], X_test, y_test, display_labels=class_names, cmap="Blues", xticks_rotation=20,
)
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

## 9. Part G — Compare Feature Importances Across Models

Random forest, gradient boosting, and XGBoost each expose `.feature_importances_`. Do they agree?

In [ ]:
importance_df = pd.DataFrame({
    "Random Forest": rf.feature_importances_,
    "Gradient Boosting": gb.feature_importances_,
    "XGBoost": xgb_model.feature_importances_,
}, index=feat_names)

importance_df.plot(kind="bar", figsize=(7, 4.5), color=["#3E5C76", "#6B4F9C", "#FF6A39"])
plt.ylabel("Importance")
plt.title("Feature importance across ensemble methods")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

importance_df

## 10. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below. Your answers must be specific to the lab results above, not just general answers from AI. 

**1. In this lab results, how did plain bagging compare to the full random forest? What does the difference (if any) tell you about the extra feature-randomness random forests add?**

*Your answer:*

**2. Boosting fits trees sequentially, each correcting the ensemble's mistakes so far. Why might that make boosting more prone to overfitting than bagging/random forests if `n_estimators` or the learning rate isn't chosen carefully?**

*Your answer:*

**3. Do your random forest, gradient boosting, and XGBoost models agree on which feature matters most? What would you tell a maintenance engineer based on this chart?**

*Your answer:*

**4. None of today's models needed a `StandardScaler`. Why not, given that Week 3's Ridge and Lasso absolutely required one?**

*Your answer:*

## 11. Deliverables & Submission

- This notebook, completed and able to run top-to-bottom without errors (`Kernel → Restart & Run All`).
- The feature-importance plot from §6, the final comparison chart and confusion matrix from §8, and the cross-model importance chart from §9.
- Your written answers to the four reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 12. Grading Rubric Guide

| Component | Weight |
|---|---|
| Bagging, random forest, gradient boosting, XGBoost all correctly trained | 35% |
| Feature importance correctly extracted and plotted (§6, §9) | 20% |
| Final comparison and interpretation (§8) | 20% |
| Reflection questions | 15% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 10% |

---
**Next week:** *Dimensionality Reduction & Clustering* — PCA and k-means/DBSCAN, for when you don't have fault labels at all.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>
